# Мини-проект по нейронным сетям: сегментация бровок карьера по ЦМР

Этот notebook оформляет проект в соответствии с требованиями задания: постановка задачи, описание данных, подготовка данных, разбиение на train/validation/test, выбор модели, обучение, метрики и минимальный код для применения обученной модели к полному DEM.


## Исходный код проекта

Проект реализован в виде отдельного репозитория на GitHub:

[CNNEdgeExtractor: сегментация бровок карьера по DEM](https://github.com/Ratven666/CNNEdgeExtractor/tree/main)

В данном notebook используется уже обученная модель и упрощённый инференс‑пайплайн поверх исходного кода репозитория.

## 1. Постановка задачи

Цель проекта — автоматически выделять **бровки карьера** на цифровой модели рельефа (DEM, ЦМР) с помощью нейронной сети.

С точки зрения машинного обучения это задача **бинарной семантической сегментации**: для каждого пикселя растра модель должна предсказать, принадлежит ли он бровке (`1`) или фону (`0`).

Проект выполнен в формате **обучения модели с нуля**: используется собственная реализация архитектуры **U-Net** на PyTorch, без дообучения внешней предобученной модели.


## 2. Описание данных

В проекте используются собственные геоданные из репозитория:

- `bd_res_1m_*.tif` — тайлы DEM с пространственным разрешением 1 м;
- `line_res_1m_*.tif` — бинарные маски бровок для соответствующих DEM-тайлов;
- `grib_1m.tif` — полный исходный DEM для итогового применения модели.

Дополнительно был рассмотрен вариант обучения не на исходной карте высот, а на **карте полных уклонов** (slope), вычисленной из DEM. Это усиливает локальные градиенты рельефа и делает бровки более контрастными для модели.


## 3. Подготовка данных

Подготовка данных включает несколько этапов:

1. Чтение DEM и mask GeoTIFF.
2. Замена `NaN` на `0`.
3. Для slope-варианта — вычисление карты уклонов по DEM через производные `dz/dx` и `dz/dy`.
4. Аугментации для train-набора: отражения, повороты, affine-преобразования, elastic transform, шум.
5. Сохранение итоговых train/val/test тайлов в отдельную структуру каталогов.

Для slope-датасета структура выглядит так:

```
data/processed_slope/
├── train/
│   ├── images/
│   └── masks/
├── val/
│   ├── images/
│   └── masks/
└── test/
    ├── images/
    └── masks/
```


## 4. Разделение на train / validation / test

Данные были разделены на три независимые части:

- `train` — обучение модели;
- `validation` — контроль качества во время обучения и выбор лучшего checkpoint;
- `test` — финальная оценка качества после завершения обучения.

В рабочем варианте использовалось разбиение:

- `train_ratio = 0.8`;
- `val_ratio = 0.1`;
- остальная часть — `test`.

Для увеличения обучающей выборки train-тайлы дополнительно аугментировались.


## 5. Выбор модели

В качестве архитектуры выбрана **U-Net**, так как это одна из базовых и наиболее подходящих CNN-архитектур для сегментации изображений и растров.

Почему U-Net подходит для задачи:

- хорошо работает на небольших датасетах;
- сочетает локальные детали и глобальный контекст за счёт encoder-decoder структуры;
- подходит для бинарной сегментации георастров;
- легко адаптируется под один входной канал (`DEM` или `slope`).

В проекте использовалась модель:

- `in_channels=1`;
- `out_channels=1`;
- функция потерь: `BCEWithLogitsLoss + DiceLoss`;
- метрики: `IoU` и `F1-score`.


## 6. Обучение модели

Обучение проводилось в PyTorch. На вход модель получала тайлы карты уклонов размером `100 x 100`, на выходе предсказывалась бинарная маска бровок.

Основные параметры обучения:

- оптимизатор: `Adam`;
- learning rate: `1e-3`;
- batch size: `4`;
- число эпох: `50`.

Во время обучения сохранялись:

- `last.ckpt` — последний checkpoint;
- `best.ckpt` — лучшая модель по `validation F1`.


## 7. Графики loss / метрик

В полноценной версии проекта стоит строить графики:

- `train loss` и `validation loss`;
- `train IoU` и `validation IoU`;
- `train F1` и `validation F1`.

В текущем практическом варианте основной акцент сделан на получении рабочего пайплайна и корректного применения модели к полному DEM. Если история обучения сохранена в CSV или логах, графики легко добавить через `matplotlib`.


## 8. Оценка на test set

Финальная оценка выполняется на `test`-выборке, которая не используется ни для обучения, ни для подбора лучшего checkpoint.

В проекте использовались метрики:

- **IoU** — пересечение предсказанной и истинной маски к их объединению;
- **F1-score** — гармоническое среднее precision и recall для бинарной сегментации.

Дополнительно качество проверялось визуально на полном DEM `grib_1m.tif` по probability map и бинарной маске.


## 9. Минимальный код для использования обученной модели

Ниже приведён минимальный код, который:

1. загружает обученную slope-модель;
2. читает полный DEM `grib_1m.tif`;
3. вычисляет slope raster;
4. применяет модель по sliding window;
5. сохраняет probability map и бинарную маску.


In [ ]:
!pip install rasterio torch

In [ ]:
import os
from pathlib import Path

import numpy as np
import rasterio
import torch
from torch import nn

from unet import UNet

In [ ]:
def compute_slope_degrees(dem: np.ndarray, res_x: float, res_y: float) -> np.ndarray:
    dem = np.nan_to_num(dem, nan=0.0).astype(np.float32)
    dz_dy, dz_dx = np.gradient(dem, res_y, res_x)
    slope_rad = np.arctan(np.sqrt(dz_dx ** 2 + dz_dy ** 2))
    slope_deg = np.degrees(slope_rad).astype(np.float32)
    return np.nan_to_num(slope_deg, nan=0.0, posinf=0.0, neginf=0.0)


def load_model(checkpoint_path: Path, device: torch.device) -> nn.Module:
    model = UNet(in_channels=1, out_channels=1)
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

In [ ]:
def normalize_patch_zscore(patch: np.ndarray) -> np.ndarray:
    patch = np.nan_to_num(patch, nan=0.0).astype(np.float32)
    mean = patch.mean()
    std = patch.std()
    if std > 1e-6:
        patch = (patch - mean) / std
    return patch.astype(np.float32)


def make_weight_window(tile_size: int) -> np.ndarray:
    yy, xx = np.mgrid[0:tile_size, 0:tile_size]
    cy, cx = (tile_size - 1) / 2.0, (tile_size - 1) / 2.0
    dist2 = (yy - cy) ** 2 + (xx - cx) ** 2
    sigma2 = (tile_size / 2.0) ** 2
    return np.exp(-dist2 / (2.0 * sigma2)).astype(np.float32)


def sliding_window_predict(model, image, tile_size=100, stride=50, threshold=0.30, device=None):
    if device is None:
        if torch.cuda.is_available():
            device = torch.device('cuda')
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = torch.device('mps')
        else:
            device = torch.device('cpu')

    h, w = image.shape
    prob_map = np.zeros((h, w), dtype=np.float32)
    weight_map = np.zeros((h, w), dtype=np.float32)
    weights = make_weight_window(tile_size)

    with torch.no_grad():
        rows = list(range(0, max(h - tile_size + 1, 1), stride))
        cols = list(range(0, max(w - tile_size + 1, 1), stride))

        if rows[-1] != h - tile_size:
            rows.append(h - tile_size)
        if cols[-1] != w - tile_size:
            cols.append(w - tile_size)

        rows = sorted(set(r for r in rows if r >= 0))
        cols = sorted(set(c for c in cols if c >= 0))

        for row in rows:
            for col in cols:
                patch = image[row:row + tile_size, col:col + tile_size]
                if patch.shape != (tile_size, tile_size):
                    continue

                patch = normalize_patch_zscore(patch)
                tensor = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(device)
                logits = model(tensor)
                probs = torch.sigmoid(logits).squeeze().cpu().numpy().astype(np.float32)

                prob_map[row:row + tile_size, col:col + tile_size] += probs * weights
                weight_map[row:row + tile_size, col:col + tile_size] += weights

    valid = weight_map > 0
    prob_map[valid] /= weight_map[valid]
    prob_map[~valid] = 0.0
    mask = (prob_map >= threshold).astype(np.uint8)
    return prob_map, mask

In [ ]:
project_root = '.'
dem_path = os.path.join(project_root, 'grib_1m.tif')
checkpoint_path = os.path.join(project_root, 'best.ckpt')
out_prob_path = Path(os.path.join(project_root, 'predictions', 'grib_1m_slope_prob.tif'))
out_mask_path = Path(os.path.join(project_root, 'predictions', 'grib_1m_slope_mask.tif'))

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = load_model(checkpoint_path, device)

with rasterio.open(dem_path) as src:
    dem = src.read(1).astype(np.float32)
    profile = src.profile.copy()
    res_x, res_y = src.res

slope = compute_slope_degrees(dem, res_x, res_y)
prob_map, mask = sliding_window_predict(model, 
                                        slope, 
                                        tile_size=100, 
                                        stride=50, 
                                        threshold=0.66, 
                                        device=device)

prob_profile = profile.copy()
prob_profile.update(dtype=rasterio.float32, count=1, compress='lzw', nodata=0.0)

mask_profile = profile.copy()
mask_profile.update(dtype=rasterio.uint8, count=1, compress='lzw', nodata=0)

out_prob_path.parent.mkdir(parents=True, exist_ok=True)
out_mask_path.parent.mkdir(parents=True, exist_ok=True)

with rasterio.open(out_prob_path, 'w', **prob_profile) as dst:
    dst.write(prob_map, 1)

with rasterio.open(out_mask_path, 'w', **mask_profile) as dst:
    dst.write(mask, 1)

print('Saved probability map to:', out_prob_path)
print('Saved binary mask to:    ', out_mask_path)

## 10. Вывод

В рамках мини-проекта была реализована и применена нейронная сеть для сегментации бровок карьера по георастрам.

Ключевые особенности проекта:

- задача относится к анализу **геоданных**;
- данные были самостоятельно подготовлены и разделены на `train/validation/test`;
- модель реализована **с нуля** в PyTorch в виде U-Net;
- для улучшения результата использовалась не только DEM, но и производный признак — **карта уклонов**;
- модель была применена к полному DEM, а результат сохранён в геопривязанном формате GeoTIFF.

Таким образом, проект полностью соответствует структуре задания и демонстрирует практическое применение нейронных сетей к задаче сегментации геопространственных данных.


<p><b>Результат модели наложенный на исходный DEM:</b></p>
<img src="img/result.png" alt="Результат сегментации" width="800">